# Media Framing Results Analysis

This notebook only analyzes already existing thesis result files.
It does not rebuild requests and it does not call the API.

Main outputs:
- run coverage summary
- overall label frequencies
- outlet-by-label counts
- outlet-by-label shares within each outlet


In [8]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd() / '2a_NER',
    Path.cwd(),
    Path.cwd().parent / '2a_NER',
    Path('/Users/MattisHaumann/Dev/Thesis/2a_NER'),
]
NOTEBOOK_DIR = next(
    (path for path in NOTEBOOK_DIR_CANDIDATES if (path / 'media_framing_batch_utils.py').exists()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate 2a_NER/media_framing_batch_utils.py from the current working directory.')

PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from media_framing_batch_utils import LEGACY_RESULT_COLUMNS, sort_by_source_order

FINAL_DIR = NOTEBOOK_DIR / 'outputs' / 'batch_media_framing' / 'thesis_final'
RUN_DIR = FINAL_DIR / 'normal_api_run'
ANALYSIS_DIR = FINAL_DIR / 'analysis'
RESULTS_PATH = RUN_DIR / 'media_framing_thesis_sync_results.csv'
ERRORS_PATH = RUN_DIR / 'media_framing_thesis_sync_errors.csv'
MANIFEST_PATH = RUN_DIR / 'media_framing_thesis_sync_manifest.csv'

ANALYSIS_SUMMARY_PATH = ANALYSIS_DIR / 'media_framing_thesis_analysis_summary.csv'
OVERALL_LABEL_SUMMARY_PATH = ANALYSIS_DIR / 'media_framing_thesis_overall_label_summary.csv'
OUTLET_LABEL_SUMMARY_PATH = ANALYSIS_DIR / 'media_framing_thesis_outlet_label_summary.csv'
OUTLET_LABEL_COUNTS_PIVOT_PATH = ANALYSIS_DIR / 'media_framing_thesis_outlet_label_counts_pivot.csv'
OUTLET_LABEL_SHARE_PIVOT_PATH = ANALYSIS_DIR / 'media_framing_thesis_outlet_label_share_pivot.csv'

CATEGORY_ORDER = [
    'POSITIONS-/PARTEILICHKEITS-BIAS',
    'VERZERRUNG/MANIPULATION',
    'DISINFORMATION/FALSCHDARSTELLUNG',
    'VERSAGEN/INKOMPETENZ',
    'NEUTRAL',
    'IRRELEVANT',
]

for required_path in [RESULTS_PATH, MANIFEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required analysis file not found: {required_path}')

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Results path: {RESULTS_PATH}')
print(f'Errors path: {ERRORS_PATH}')
print(f'Manifest path: {MANIFEST_PATH}')


Results path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/normal_api_run/media_framing_thesis_sync_results.csv
Errors path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/normal_api_run/media_framing_thesis_sync_errors.csv
Manifest path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/normal_api_run/media_framing_thesis_sync_manifest.csv


## 1. Load and Validate the Existing Result Files


In [9]:
results_df = pd.read_csv(RESULTS_PATH)
errors_df = pd.read_csv(ERRORS_PATH) if ERRORS_PATH.exists() and ERRORS_PATH.stat().st_size > 0 else pd.DataFrame()
manifest_df = pd.read_csv(MANIFEST_PATH)

if not results_df.empty:
    results_df = results_df[LEGACY_RESULT_COLUMNS].copy()
    results_df['source'] = results_df['source'].fillna('').astype(str)
    results_df['category'] = pd.Categorical(results_df['category'], categories=CATEGORY_ORDER, ordered=True)
    results_df = results_df.sort_values(
        ['row_id', 'context_idx', 'source', 'hit_text'],
        ascending=[True, True, True, True],
    ).drop_duplicates(subset=['hit_id'], keep='last').reset_index(drop=True)
    if results_df['hit_id'].duplicated().any():
        raise AssertionError('Duplicate hit_id values found in results_df.')

analysis_summary_df = pd.DataFrame(
    [
        {
            'manifest_rows': len(manifest_df),
            'result_rows': len(results_df),
            'error_rows': len(errors_df),
            'remaining_rows_without_success': max(len(manifest_df) - len(results_df), 0),
            'unique_articles_in_results': results_df['row_id'].nunique() if not results_df.empty else 0,
            'unique_outlets_in_results': results_df['source'].nunique() if not results_df.empty else 0,
        }
    ]
)
analysis_summary_df.to_csv(ANALYSIS_SUMMARY_PATH, index=False, encoding='utf-8')

print(f'Analysis summary written to: {ANALYSIS_SUMMARY_PATH}')
display(analysis_summary_df)
display(results_df.head(5))
display(errors_df.head(5))


Analysis summary written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/analysis/media_framing_thesis_analysis_summary.csv


,manifest_rows,result_rows,error_rows,remaining_rows_without_success,unique_articles_in_results,unique_outlets_in_results
0,11975,11973,2,2,7204,7


,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window,model,response_id,category,evidence,raw_response_json
0,d844fe7dbcb4b28a,1,Antispiegel,Bereitet der Westen die Entmachtung oder sogar...,Politico,1,1,1,"Er hat es geschafft, seine Leute überall zu pl...",gpt-5-mini,resp_0e3ba66f85f590500069c4635798988194bf13317...,NEUTRAL,NaN,"{""id"": ""resp_0e3ba66f85f590500069c463579898819..."
1,f785774b2205e418,4,Antispiegel,Fordert Russland wirklich die Vernichtung alle...,Bild-Zeitung,1,1,1,Viele haben Reitschuster aus der Corona-Zeit j...,gpt-5-mini,resp_05d159969059d7ad0069c4635784cc81938df21e9...,DISINFORMATION/FALSCHDARSTELLUNG,Propagandisten von Bild-Zeitung,"{""id"": ""resp_05d159969059d7ad0069c4635784cc819..."
2,8beba1b1263837c4,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,1,1,1,Der Spiegel macht mal wieder Berichterstattung...,gpt-5-mini,resp_084a85555116e7270069c4635799bc81949029652...,VERSAGEN/INKOMPETENZ,Berichterstattung für den Kindergarten,"{""id"": ""resp_084a85555116e7270069c4635799bc819..."
3,db28e215f2d3989d,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,2,1,1,Das NABU ist das vielleicht wichtigste Instrum...,gpt-5-mini,resp_067c239f074dd8230069c46357985c8193befbad9...,NEUTRAL,NaN,"{""id"": ""resp_067c239f074dd8230069c46357985c819..."
4,387a8dd7ac2cedde,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,3,4,1,"Ich denke, die Antwort liegt auf der Hand. Mär...",gpt-5-mini,resp_0f30c6857a242fb20069c463578cd0819496d5c8e...,POSITIONS-/PARTEILICHKEITS-BIAS,anti-russischen und pro-ukrainischen Propagand...,"{""id"": ""resp_0f30c6857a242fb20069c463578cd0819..."


,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window,status_code,error_type,error_payload,raw_response_json
0,97bf745e6b2d9b17,6902,RT_de,Werden Moskau und Washington Ukraine-Gespräche...,Politico,2,2,1,Ein namentlich nicht genannter US‑Beamter teil...,520,http_error,"{""message"": ""<!DOCTYPE html>\n<!--[if lt IE 7]...",NaN
1,1edca253aff4252e,6903,RT_de,Erste Reaktionen in Berlin auf US-Friedensplan...,RTL,1,1,1,Der Kanzleramtschef Thorsten Frei hat sich übe...,520,http_error,"{""message"": ""<!DOCTYPE html>\n<!--[if lt IE 7]...",NaN


## 2. Overall Label Frequencies


In [10]:
if results_df.empty:
    print('No coded results available yet.')
else:
    overall_label_summary_df = (
        results_df.groupby('category', as_index=False, observed=False)
        .agg(
            coded_contexts=('hit_id', 'size'),
            unique_articles=('row_id', 'nunique'),
            unique_outlets=('source', 'nunique'),
            avg_hits_per_context=('count_hits', 'mean'),
            avg_unique_entities_per_context=('count_unique_entities', 'mean'),
        )
        .sort_values('category')
        .reset_index(drop=True)
    )
    overall_label_summary_df['share_pct'] = (
        overall_label_summary_df['coded_contexts'] / overall_label_summary_df['coded_contexts'].sum() * 100
    ).round(2)
    overall_label_summary_df['avg_hits_per_context'] = overall_label_summary_df['avg_hits_per_context'].round(2)
    overall_label_summary_df['avg_unique_entities_per_context'] = overall_label_summary_df['avg_unique_entities_per_context'].round(2)
    overall_label_summary_df.to_csv(OVERALL_LABEL_SUMMARY_PATH, index=False, encoding='utf-8')

    print(f'Overall label summary written to: {OVERALL_LABEL_SUMMARY_PATH}')
    display(overall_label_summary_df)


Overall label summary written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/analysis/media_framing_thesis_overall_label_summary.csv


,category,coded_contexts,unique_articles,unique_outlets,avg_hits_per_context,avg_unique_entities_per_context,share_pct
0,POSITIONS-/PARTEILICHKEITS-BIAS,1409,964,7,2.62,1.62,11.77
1,VERZERRUNG/MANIPULATION,1084,764,7,2.33,1.45,9.05
2,DISINFORMATION/FALSCHDARSTELLUNG,377,292,7,3.07,1.50,3.15
3,VERSAGEN/INKOMPETENZ,322,265,7,2.32,1.39,2.69
4,NEUTRAL,8568,6015,7,1.29,1.12,71.56
5,IRRELEVANT,213,187,7,1.24,1.00,1.78


## 3. Compare Labels Across Outlets

The long table keeps both counts and within-outlet shares.
The pivot tables are easier to scan for thesis reporting.


In [11]:
if results_df.empty:
    print('No coded results available yet.')
else:
    outlet_totals_df = (
        results_df.groupby('source', as_index=False)
        .agg(
            outlet_total_contexts=('hit_id', 'size'),
            outlet_total_articles=('row_id', 'nunique'),
            avg_hits_per_context=('count_hits', 'mean'),
            avg_unique_entities_per_context=('count_unique_entities', 'mean'),
        )
        .pipe(sort_by_source_order)
    )
    outlet_totals_df['avg_hits_per_context'] = outlet_totals_df['avg_hits_per_context'].round(2)
    outlet_totals_df['avg_unique_entities_per_context'] = outlet_totals_df['avg_unique_entities_per_context'].round(2)

    outlet_label_summary_df = (
        results_df.groupby(['source', 'category'], as_index=False, observed=False)
        .agg(
            coded_contexts=('hit_id', 'size'),
            unique_articles_with_label=('row_id', 'nunique'),
        )
        .merge(outlet_totals_df, on='source', how='left')
    )
    outlet_label_summary_df['share_within_outlet_pct'] = (
        outlet_label_summary_df.groupby('source')['coded_contexts']
        .transform(lambda values: (values / values.sum() * 100).round(2))
    )
    outlet_label_summary_df = (
        outlet_label_summary_df
        .pipe(sort_by_source_order)
        .sort_values(['source', 'category'])
        .reset_index(drop=True)
    )

    outlet_label_counts_pivot_df = (
        outlet_label_summary_df.pivot(index='source', columns='category', values='coded_contexts')
        .fillna(0)
        .astype(int)
        .reset_index()
        .pipe(sort_by_source_order)
    )
    outlet_label_share_pivot_df = (
        outlet_label_summary_df.pivot(index='source', columns='category', values='share_within_outlet_pct')
        .fillna(0)
        .reset_index()
        .pipe(sort_by_source_order)
    )

    outlet_label_summary_df.to_csv(OUTLET_LABEL_SUMMARY_PATH, index=False, encoding='utf-8')
    outlet_label_counts_pivot_df.to_csv(OUTLET_LABEL_COUNTS_PIVOT_PATH, index=False, encoding='utf-8')
    outlet_label_share_pivot_df.to_csv(OUTLET_LABEL_SHARE_PIVOT_PATH, index=False, encoding='utf-8')

    print(f'Outlet-label summary written to: {OUTLET_LABEL_SUMMARY_PATH}')
    print(f'Outlet-label count pivot written to: {OUTLET_LABEL_COUNTS_PIVOT_PATH}')
    print(f'Outlet-label share pivot written to: {OUTLET_LABEL_SHARE_PIVOT_PATH}')
    display(outlet_label_summary_df)
    display(outlet_label_counts_pivot_df)
    display(outlet_label_share_pivot_df)


Outlet-label summary written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/analysis/media_framing_thesis_outlet_label_summary.csv
Outlet-label count pivot written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/analysis/media_framing_thesis_outlet_label_counts_pivot.csv
Outlet-label share pivot written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/thesis_final/analysis/media_framing_thesis_outlet_label_share_pivot.csv


,source,category,coded_contexts,unique_articles_with_label,outlet_total_contexts,outlet_total_articles,avg_hits_per_context,avg_unique_entities_per_context,share_within_outlet_pct
0,Antispiegel,POSITIONS-/PARTEILICHKEITS-BIAS,75,51,540,221,2.56,1.09,13.89
1,Antispiegel,VERZERRUNG/MANIPULATION,163,82,540,221,2.56,1.09,30.19
2,Antispiegel,DISINFORMATION/FALSCHDARSTELLUNG,45,32,540,221,2.56,1.09,8.33
3,Antispiegel,VERSAGEN/INKOMPETENZ,26,19,540,221,2.56,1.09,4.81
4,Antispiegel,NEUTRAL,223,138,540,221,2.56,1.09,41.30
5,Antispiegel,IRRELEVANT,8,8,540,221,2.56,1.09,1.48
6,Compact,POSITIONS-/PARTEILICHKEITS-BIAS,81,62,512,356,1.36,1.15,15.82
7,Compact,VERZERRUNG/MANIPULATION,58,50,512,356,1.36,1.15,11.33
8,Compact,DISINFORMATION/FALSCHDARSTELLUNG,34,29,512,356,1.36,1.15,6.64
9,Compact,VERSAGEN/INKOMPETENZ,9,9,512,356,1.36,1.15,1.76


category,source,POSITIONS-/PARTEILICHKEITS-BIAS,VERZERRUNG/MANIPULATION,DISINFORMATION/FALSCHDARSTELLUNG,VERSAGEN/INKOMPETENZ,NEUTRAL,IRRELEVANT
0,Antispiegel,75,163,45,26,223,8
1,Compact,81,58,34,9,283,47
2,Deutschlandkurier,94,21,18,5,242,1
3,Nius,362,296,108,91,1127,42
4,RT_de,172,160,43,23,1433,31
5,Tagesschau,31,13,5,5,4281,35
6,Tichys_Einblick,594,373,124,163,979,49


category,source,POSITIONS-/PARTEILICHKEITS-BIAS,VERZERRUNG/MANIPULATION,DISINFORMATION/FALSCHDARSTELLUNG,VERSAGEN/INKOMPETENZ,NEUTRAL,IRRELEVANT
0,Antispiegel,13.89,30.19,8.33,4.81,41.30,1.48
1,Compact,15.82,11.33,6.64,1.76,55.27,9.18
2,Deutschlandkurier,24.67,5.51,4.72,1.31,63.52,0.26
3,Nius,17.87,14.61,5.33,4.49,55.63,2.07
4,RT_de,9.24,8.59,2.31,1.24,76.96,1.66
5,Tagesschau,0.71,0.30,0.11,0.11,97.96,0.80
6,Tichys_Einblick,26.03,16.35,5.43,7.14,42.90,2.15


## 4. Optional Inspection Table

Use this to inspect one outlet or one label without touching the run notebook.


In [12]:
FILTER_SOURCE = None
FILTER_CATEGORY = None

if results_df.empty:
    print('No coded results available yet.')
else:
    inspection_df = results_df.copy()
    if FILTER_SOURCE:
        inspection_df = inspection_df[inspection_df['source'] == FILTER_SOURCE].copy()
    if FILTER_CATEGORY:
        inspection_df = inspection_df[inspection_df['category'] == FILTER_CATEGORY].copy()

    inspection_columns = [
        'row_id',
        'source',
        'Title',
        'hit_text',
        'category',
        'evidence',
        'count_hits',
        'count_unique_entities',
        'context_window',
    ]
    inspection_df = inspection_df[inspection_columns].sort_values(
        ['source', 'category', 'row_id', 'hit_text'],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

    print(f'Inspection rows: {len(inspection_df):,}')
    display(inspection_df.head(50))


Inspection rows: 11,973


,row_id,source,Title,hit_text,category,evidence,count_hits,count_unique_entities,context_window
0,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,anti-russischen und pro-ukrainischen Propagand...,4,1,"Ich denke, die Antwort liegt auf der Hand. Mär..."
1,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,vom Westen orchestrierten Putsche,4,1,Wird Selenskys Absetzung medial vorbereitet? A...
2,42,Antispiegel,Der Spiegel und „Trumps Märchen von der Obama-...,Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,gleichgeschaltet,1,1,Der Spiegel und „Trumps Märchen von der Obama-...
3,46,Antispiegel,"Mein neues Buch „Gesteuerte Wahrheit“ erklärt,...",Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,wer sie mit welchen Mitteln lenkt,2,1,Nach einigen Jahren Pause habe ich endlich wie...
4,61,Antispiegel,"Drei aktuelle Spiegel-Artikel zeigen, wie deut...",Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,Selensky hat immer Recht,1,1,Und während die Staaten des Westens Schulden m...
5,68,Antispiegel,Die EU hätte sich ein Beispiel an Indien nehme...,Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,Pressesprecher der EU-Kommission,3,1,Trotzdem geben sich die angeblich so kritische...
6,69,Antispiegel,Der Spiegel und die Gesundheit von US-Präsidenten,Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,Kampagne im Spiegel,7,1,Der Spiegel und die Gesundheit von US-Präsiden...
7,70,Antispiegel,Was deutsche Medien über den Streit zwischen d...,Reuters | Spiegel | Tagesschau,POSITIONS-/PARTEILICHKEITS-BIAS,Schweigekartell,5,3,Das ukrainische Portal „Evropeiskaja Prawda“ b...
8,71,Antispiegel,Warum Trump die Nationalgarde in die US-Bundes...,Spiegel,POSITIONS-/PARTEILICHKEITS-BIAS,den US-Demokraten treu ergebenen Medien,1,1,"Der Grund für Trump, die Nationalgarde, die ei..."
9,75,Antispiegel,In der Ukraine wurde der Neonazismus als Staat...,Staatssender,POSITIONS-/PARTEILICHKEITS-BIAS,Staatssender,1,1,"Zumindest nicht auf Deutsch, dafür aber auf Ru..."


In [ ]:
from pathlib import Path
import pandas as pd

# join dates from df_combined.csv into results_df by 'row_id'
COMBINED_PATH = Path("/Users/MattisHaumann/Dev/Thesis/2a_NER/df_combined.csv")
if not COMBINED_PATH.exists():
    raise FileNotFoundError(f"Combined file not found: {COMBINED_PATH}")

df_combined = pd.read_csv(COMBINED_PATH)

if "row_id" not in df_combined.columns:
    raise KeyError(
        f"df_combined.csv must contain a 'row_id' column. "
        f"Available columns: {list(df_combined.columns)}"
    )

# normalize row_id to string to avoid type mismatch on merge
results_df = results_df.copy()
results_df["row_id"] = results_df["row_id"].astype(str)
df_combined["row_id"] = df_combined["row_id"].astype(str)

# detect a date-like column in df_combined
date_candidates = [
    c for c in df_combined.columns
    if c.lower() in {
        "date", "published", "publish_date", "publication_date",
        "pub_date", "published_at", "timestamp", "datetime"
    }
]
if not date_candidates:
    date_candidates = [
        c for c in df_combined.columns
        if any(k in c.lower() for k in ("date", "time", "published"))
    ]

if not date_candidates:
    raise KeyError(f"No date-like column found in {COMBINED_PATH}. Columns: {list(df_combined.columns)}")

date_col = date_candidates[0]

# keep merge safe: one date per row_id
join_df = df_combined[["row_id", date_col]].drop_duplicates()
duplicate_row_ids = join_df.loc[join_df["row_id"].duplicated(), "row_id"].unique()
if len(duplicate_row_ids) > 0:
    raise ValueError(
        f"{COMBINED_PATH} has duplicate row_id values in '{date_col}', "
        "which would duplicate rows in results_df."
    )

if "date" in results_df.columns:
    results_df = results_df.drop(columns=["date"])

results_df = results_df.merge(
    join_df.rename(columns={date_col: "date"}),
    on="row_id",
    how="left",
)

results_df["date"] = pd.to_datetime(results_df["date"], errors="coerce")

print(f"Joined date column '{date_col}' from {COMBINED_PATH}")
print(f"Missing/failed-to-parse dates: {results_df['date'].isna().sum()} of {len(results_df)} rows")
display(results_df[["row_id", "date"]].drop_duplicates().head(20))

NameError: name 'results_df' is not defined